# Grasp Detection Training (Colab)

This notebook runs training and evaluation on Colab GPU.

**Workflow:**
1. Mount Google Drive (persistent storage for dataset + checkpoints)
2. Clone or pull the project repo from GitHub
3. Symlink `data/` and `results/` to Drive so they survive session disconnects
4. Download Cornell Grasping Dataset (first run only)
5. Run training and evaluation

**Before running:**
- Set runtime to GPU: `Runtime → Change runtime type → T4 GPU`
- Edit `GITHUB_REPO_URL` in the next cell to point to your repo

In [ ]:
# ===== CONFIGURATION — EDIT THIS =====
GITHUB_REPO_URL = "https://github.com/bayerworld233-svg/grasp-detection-paper.git"
PROJECT_NAME = "grasp-detection-paper"
DRIVE_BASE = "/content/drive/MyDrive/grasp-detection"

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive mounted. Base dir: {DRIVE_BASE}")

In [ ]:
# Clone repo (first time) or pull latest changes (subsequent runs)
import os, subprocess

PROJECT_DIR = f"/content/{PROJECT_NAME}"
if os.path.exists(PROJECT_DIR):
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")
!ls

In [ ]:
# Install dependencies (Colab has torch/torchvision preinstalled, only the extras matter)
!pip install -q -r requirements.txt

In [ ]:
# Symlink data/ and results/ to Drive so they persist across sessions
import os, shutil

DATA_DRIVE = f"{DRIVE_BASE}/data"
RESULTS_DRIVE = f"{DRIVE_BASE}/results"
os.makedirs(DATA_DRIVE, exist_ok=True)
os.makedirs(RESULTS_DRIVE, exist_ok=True)

for local, drive_path in [("data", DATA_DRIVE), ("results", RESULTS_DRIVE)]:
    local_path = os.path.join(PROJECT_DIR, local)
    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.exists(local_path):
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)

print("Symlinked data/ and results/ to Drive")
!ls -la | grep -E '(data|results)'

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"PyTorch: {torch.__version__}")

In [ ]:
# Load Kaggle API token from Colab Secrets (left sidebar 🔑 → KAGGLE_API_TOKEN)
from google.colab import userdata
import os
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Kaggle token loaded from Colab Secrets")

In [ ]:
# Download Cornell Grasping Dataset (skips if already on Drive)
!bash scripts/download_data.sh

In [ ]:
# Train ResNet-50 with ImageNet pretrained weights
!python -m src.train --pretrained --output results/pretrained

In [ ]:
# Train ResNet-50 from scratch
!python -m src.train --output results/scratch

In [ ]:
# Evaluate both checkpoints with the Cornell metric
!python -m src.evaluate --checkpoint results/pretrained/best.pth
!python -m src.evaluate --checkpoint results/scratch/best.pth

In [ ]:
# Generate qualitative visualizations on test set (sample images with predicted vs ground-truth grasps)
!python -m src.evaluate --checkpoint results/pretrained/best.pth --visualize --num_samples 8